# SignSage · FDMSE-ISL training v2 (CPU-friendly, resumable)

Trains an isolated-word recogniser on the features produced by **notebook 01** (cell 12, folder `features_v4_pose_face`). It reads only `features/feat_chunk_#####.npz` and `meta/manifest.csv` - no videos, no MediaPipe - so it is small enough for an old laptop (2 cores, 8 GB RAM) and a GPU is not needed.

**Cells:** 1 Settings · 2 Setup · 3 write code · 4 Load data · **5 Train (resumable)** · 6 Test once + confidence threshold + safety checks · 7 Final retrain on all signers (optional) · 8 Export

**Rules built in**
- Reported accuracy comes from the **official signer-independent split**: the model is chosen on validation and the **test split is evaluated once** (a saved report blocks accidental re-testing).
- A checkpoint is saved after every epoch; rerun cell 5 to continue after any interruption.
- Cell 4 prints exactly how many clips were used per split and how many were unusable - report these numbers in the thesis.
- Settings and data are fingerprinted: resuming with different settings or newly converted data is refused (use a new `RUN_NAME`).
- Cell 4 also checks how well MediaPipe saw the clips (pose, hands, **face**). If it warns, stop and fix that first.
- If extraction is not finished, you can still train to test the pipeline, but the numbers are **not final**.

In [ ]:
# ============================ 1. SETTINGS (edit only here) ============================
RUN_NAME        = "baseline_v1"   # new name = new run (own folder, own checkpoints)
OUT_ROOT        = None            # None -> the same folder notebook 01 used (Drive on Colab, ./signsage_fdmse_v2 elsewhere)
FEATURE_SET     = "v4_pose_face"  # which features_<name> folder notebook 01 (cell 12) built. Do NOT use the old "features" folder:
                                  # its face points are empty (the face model never found a face)

EPOCHS          = 40              # upper limit; early stopping usually ends sooner
BATCH           = 128
LR              = 2e-3
WEIGHT_DECAY    = 1e-2
HIDDEN          = 128             # BiGRU size (128 is plenty for an 8 GB laptop)
LAYERS          = 2
DROPOUT         = 0.3
LABEL_SMOOTHING = 0.1
PATIENCE        = 8               # stop after this many epochs without validation improvement
NOISE, MASK_P, SCALE_LO = 0.05, 0.10, 0.8   # augmentation strength (noise, frame dropout, speed jitter)

THREADS         = None            # CPU threads for PyTorch; None = half the logical cores (i3-7020U: 2). Try 4 and compare epoch time.
SEED            = 42
TARGET_PRECISION = 0.90           # confidence threshold = smallest value whose accepted predictions are this precise (on validation)
MAX_TRAIN_CLIPS = None            # e.g. 3000 for a 2-minute trial run
RUN_FINAL_RETRAIN = True          # cell 7: retrain on ALL signers for deployment
FORCE_RETEST    = False           # keep False: the test split is meant to be used once

In [ ]:
# ============================ 2. SETUP ============================
import os, sys, json, importlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = OUT_ROOT or "/content/drive/MyDrive/SignSage_FDMSE_v2"
else:
    OUT_ROOT = OUT_ROOT or os.path.abspath("./signsage_fdmse_v2")
try:
    import torch
except ImportError:
    raise SystemExit("PyTorch is not installed. On the laptop (inside the venv):\n"
                     "  uv pip install torch --index-url https://download.pytorch.org/whl/cpu\n"
                     "See LAPTOP_RUN_GUIDE.md.")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
n_threads = THREADS or max(1, (os.cpu_count() or 2) // 2)
torch.set_num_threads(n_threads)

META, FEATS = f"{OUT_ROOT}/meta", f"{OUT_ROOT}/features_{FEATURE_SET}"
RUN_DIR = f"{OUT_ROOT}/training/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
for need in (f"{META}/manifest.csv", f"{META}/label_map.json"):
    if not os.path.exists(need):
        raise SystemExit(f"Missing {need}. Run notebook 01 first (cells 8 and 12).")
if not os.path.exists(f"{FEATS}/feature_spec.json"):
    raise SystemExit(f"{FEATS} not found. Run notebook 01 cell 12 (it builds features_{FEATURE_SET}).")
FEATURE_SPEC = json.load(open(f"{FEATS}/feature_spec.json"))
print(f"features: {FEATURE_SPEC['name']} | face_source={FEATURE_SPEC['face_source']} | aspect={FEATURE_SPEC['aspect']}")
print(f"torch {torch.__version__} | threads {n_threads} | cpu cores {os.cpu_count()} | run folder: {RUN_DIR}")

In [ ]:
%%writefile sp_train.py
"""SignSage FDMSE-ISL training (v2) - built for a small CPU (2 cores, 8 GB RAM).

* Reads only the saved feature chunks (feat_chunk_#####.npz) + the manifest: no video, no detectors.
* Resumable: a checkpoint is written atomically after every epoch.
* The test split is evaluated ONCE per run directory (a saved report guards against re-testing).
* Exports a self-describing bundle (weights, preprocessing, labels, metrics, model card).
"""
from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import zipfile
import math
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

FEAT_RE = re.compile(r"^feat_chunk_(\d{5})\.npz$")
N_FRAMES, FEAT_DIM = 60, 122


# --------------------------------------------------------------------------
# data
# --------------------------------------------------------------------------
def _load_npz_retry(path, retries=6, log=print):
    """Copy to local disk first and retry: Google Drive's mount occasionally drops a read."""
    last = None
    for a in range(retries):
        tmp = None
        try:
            fd, tmp = tempfile.mkstemp(suffix=".npz")
            os.close(fd)
            shutil.copyfile(path, tmp)
            with np.load(tmp, allow_pickle=False) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, ValueError, EOFError, KeyError) as e:
            last = e
            log(f"  read retry {a+1}/{retries} for {Path(path).name}: {e.__class__.__name__}")
            time.sleep(min(2 ** (a + 1), 30))
        finally:
            if tmp and os.path.exists(tmp):
                os.unlink(tmp)
    raise SystemExit(f"Could not read {path} after {retries} tries ({last.__class__.__name__}). "
                     "Rerun the cell; if it persists remount Drive (restart the runtime and run cell 2).")


def load_dataset(feat_dir, manifest, log=print):
    """Return dict(X float16 (N,60,122), y int64, split str, signer str, stats)."""
    feat_dir = Path(feat_dir)
    files = sorted(p for p in feat_dir.iterdir() if FEAT_RE.match(p.name)) if feat_dir.is_dir() else []
    if not files:
        raise SystemExit(f"No feature files in {feat_dir}. Run the extraction notebook (cells 10 and 12) first.")
    key_to_row = {hashlib.sha1(p.encode()).hexdigest(): i for i, p in enumerate(manifest["path"])}
    cls = manifest["class_id"].to_numpy()
    spl = manifest["split"].to_numpy()
    sig = manifest["signer"].to_numpy()
    Xs, rows_ok, rows_seen, qs = [], [], [], []
    for p in files:
        z = _load_npz_retry(p, log=log)
        keys = z["keys"].astype(str)
        rows = np.array([key_to_row.get(k, -1) for k in keys], dtype=np.int64)
        known = rows >= 0
        rows_seen.append(rows[known])
        sel = known & z["usable"].astype(bool)
        Xs.append(z["X"][sel].astype(np.float16))
        qs.append(z["quality"][sel].astype(np.float32))
        rows_ok.append(rows[sel])
    X = np.concatenate(Xs)
    rows_ok = np.concatenate(rows_ok)
    rows_seen = np.concatenate(rows_seen)
    if X.shape[1:] != (N_FRAMES, FEAT_DIM):
        raise SystemExit(f"Unexpected feature shape {X.shape[1:]}; expected {(N_FRAMES, FEAT_DIM)}")
    out = dict(X=X, y=cls[rows_ok].astype(np.int64), split=spl[rows_ok], signer=sig[rows_ok], rows=rows_ok,
               quality=np.concatenate(qs))
    n_cls = int(manifest["class_id"].max()) + 1
    seen_split = pd.Series(spl[rows_seen]).value_counts().to_dict()
    ok_split = pd.Series(out["split"]).value_counts().to_dict()
    stats = dict(n_classes=n_cls, converted=int(len(rows_seen)), usable=int(len(rows_ok)),
                 per_split_converted={k: int(v) for k, v in seen_split.items()},
                 per_split_usable={k: int(v) for k, v in ok_split.items()},
                 total_in_manifest=int(len(manifest)), feature_files=len(files))
    for sp in ("train", "validation", "test"):
        m = out["split"] == sp
        stats[f"classes_in_{sp}"] = int(len(np.unique(out["y"][m]))) if m.any() else 0
    out["stats"] = stats
    log(f"feature files: {len(files)} | clips converted: {stats['converted']}/{stats['total_in_manifest']} "
        f"| usable: {stats['usable']}")
    for sp in ("train", "validation", "test"):
        c, u = seen_split.get(sp, 0), ok_split.get(sp, 0)
        log(f"  {sp:<10} converted {c:>6} | usable {u:>6} | unusable {c-u:>5} | classes present {stats[f'classes_in_{sp}']}/{n_cls}")
    return out


def quality_report(data, std=None, log=print):
    """How well MediaPipe saw each clip, and whether any feature is dead. Returns a list of warnings."""
    q = data["quality"]
    warnings = []
    log("detection quality over all usable clips (fraction of frames):")
    for j, name in enumerate(("pose detected", "hands visible", "face detected")):
        v = q[:, j]
        log(f"  {name:<14} mean {v.mean():.3f} | median {np.median(v):.3f} | 5th percentile {np.percentile(v, 5):.3f}"
            f" | clips below 0.5: {(v < 0.5).mean():.1%}")
    if np.median(q[:, 2]) < 0.8:
        warnings.append(
            "FACE was detected in fewer than 80% of frames (median). The hand-to-face features would be built "
            "from missing face points, and the live app (which sees real faces) would then give the model "
            "inputs it never saw in training. Do not trust the results until this is fixed - see the notes in "
            "the notebook.")
    if np.median(q[:, 1]) < 0.7:
        warnings.append("Hands are visible in fewer than 70% of frames (median): check the hand detection.")
    if std is not None:
        dead = np.flatnonzero(std <= 1.0001e-3)
        log(f"feature columns with no variation (dead): {len(dead)} of {FEAT_DIM}" + (f" -> {dead.tolist()}" if len(dead) else ""))
        if len(dead) > 10:
            warnings.append(f"{len(dead)} of {FEAT_DIM} features never change - part of the input is empty.")
    for w in warnings:
        log("WARNING: " + w)
    if not warnings:
        log("quality check passed (no warnings).")
    return warnings


def fit_standardizer(X_train):
    """Per-feature mean/std over every frame of the training clips (float32 accumulation)."""
    n = 0
    s = np.zeros(FEAT_DIM, np.float64)
    ss = np.zeros(FEAT_DIM, np.float64)
    for i in range(0, len(X_train), 2048):
        b = X_train[i:i + 2048].astype(np.float32).reshape(-1, FEAT_DIM)
        s += b.sum(0)
        ss += (b.astype(np.float64) ** 2).sum(0)
        n += len(b)
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 1e-12))
    return mean.astype(np.float32), np.maximum(std, 1e-3).astype(np.float32)


# --------------------------------------------------------------------------
# model
# --------------------------------------------------------------------------
class SignGRU(nn.Module):
    """Linear projection -> 2-layer BiGRU -> attention + max pooling -> classifier."""

    def __init__(self, n_classes, in_dim=FEAT_DIM, proj=128, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, proj), nn.GELU(), nn.Dropout(dropout * 0.5))
        self.gru = nn.GRU(proj, hidden, num_layers=layers, batch_first=True, bidirectional=True,
                          dropout=dropout if layers > 1 else 0.0)
        self.attn = nn.Linear(2 * hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * hidden, n_classes))

    def forward(self, x):
        h, _ = self.gru(self.proj(x))
        w = torch.softmax(self.attn(h).squeeze(-1), dim=1).unsqueeze(-1)
        return self.head(torch.cat([(h * w).sum(1), h.max(1).values], dim=1))


def build_model(arch):
    return SignGRU(arch["n_classes"], FEAT_DIM, arch["proj"], arch["hidden"], arch["layers"], arch["dropout"])


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def augment(x, gen, noise=0.05, mask_p=0.10, scale_lo=0.8):
    """Time-crop-and-stretch (speed jitter), Gaussian noise and random frame dropout."""
    b, t, _ = x.shape
    s = float(torch.empty(1).uniform_(scale_lo, 1.0, generator=gen))
    length = max(8, int(round(t * s)))
    if length < t:
        start = int(torch.randint(0, t - length + 1, (1,), generator=gen))
        x = F.interpolate(x[:, start:start + length].transpose(1, 2), size=t, mode="linear",
                          align_corners=False).transpose(1, 2)
    if noise:
        x = x + noise * torch.randn(x.shape, generator=gen)
    if mask_p:
        x = x.masked_fill(torch.rand(b, t, 1, generator=gen) < mask_p, 0.0)
    return x


def lr_at(step, total, base, warmup):
    if step < warmup:
        return base * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))


# --------------------------------------------------------------------------
# evaluation helpers
# --------------------------------------------------------------------------
@torch.no_grad()
def predict_probs(model, X, mean, std, batch=512):
    model.eval()
    m, s = torch.from_numpy(mean), torch.from_numpy(std)
    out = []
    for i in range(0, len(X), batch):
        xb = (torch.from_numpy(X[i:i + batch]).float() - m) / s
        out.append(torch.softmax(model(xb), dim=1).numpy())
    return np.concatenate(out) if out else np.zeros((0, 1), np.float32)


def metrics_from_probs(probs, y, n_classes):
    pred = probs.argmax(1)
    top1 = float((pred == y).mean())
    k = min(5, probs.shape[1])
    topk = np.argpartition(-probs, k - 1, axis=1)[:, :k]
    top5 = float((topk == y[:, None]).any(1).mean())
    nll = float(-np.log(np.maximum(probs[np.arange(len(y)), y], 1e-9)).mean())
    recalls = []
    for c in np.unique(y):
        recalls.append(float((pred[y == c] == c).mean()))
    return dict(top1=top1, top5=top5, loss=nll, macro_recall=float(np.mean(recalls)),
                n=int(len(y)), n_classes_present=int(len(np.unique(y))))


def per_class_report(probs, y, classes):
    pred = probs.argmax(1)
    rows = []
    for c in np.unique(y):
        m = y == c
        wrong = pred[m][pred[m] != c]
        conf = int(np.bincount(wrong).argmax()) if len(wrong) else -1
        rows.append(dict(class_id=int(c), word=classes[c], n=int(m.sum()), recall=float((pred[m] == c).mean()),
                         most_confused_with=classes[conf] if conf >= 0 else ""))
    return pd.DataFrame(rows).sort_values("recall")


def calibrate_threshold(probs, y, target_precision=0.90, min_accepted=50, floor=0.02):
    """Smallest confidence threshold (not below `floor`) whose accepted predictions reach
    `target_precision` on the given (validation) data. A correct model can still be modest in
    confidence, so no large fixed floor is imposed."""
    conf = probs.max(1)
    pred = probs.argmax(1)
    ok = pred == y
    table = []
    for t in np.round(np.arange(0.05, 0.99, 0.05), 2):
        acc = conf >= t
        table.append(dict(threshold=float(t), coverage=float(acc.mean()),
                          precision=float(ok[acc].mean()) if acc.any() else float("nan")))
    chosen = None
    for t in np.round(np.arange(floor, 1.0, 0.01), 2):
        acc = conf >= t
        if acc.sum() >= min_accepted and ok[acc].mean() >= target_precision:
            chosen = float(t)
            break
    return chosen, pd.DataFrame(table)


def rejection_checks(model, mean, std, X_ref, threshold, seed=0):
    """Inputs that are NOT signs must stay below the confidence threshold."""
    g = np.random.RandomState(seed)
    m, s = torch.from_numpy(mean), torch.from_numpy(std)
    ref = X_ref[g.randint(len(X_ref))].astype(np.float32)
    cases = {
        "all zeros": np.zeros((1, N_FRAMES, FEAT_DIM), np.float32),
        "random noise": g.randn(1, N_FRAMES, FEAT_DIM).astype(np.float32) * std + mean,
        "frozen pose (one frame repeated)": np.repeat(ref[:1][None], N_FRAMES, axis=1),
        "5 frames only (rest zeros)": np.concatenate([ref[:5], np.zeros((N_FRAMES - 5, FEAT_DIM), np.float32)])[None],
    }
    model.eval()
    res = {}
    with torch.no_grad():
        for name, arr in cases.items():
            p = torch.softmax(model((torch.from_numpy(arr) - m) / s), dim=1)[0]
            res[name] = dict(max_confidence=float(p.max()),
                             rejected=bool(threshold is not None and float(p.max()) < threshold))
    return res


# --------------------------------------------------------------------------
# training (resumable)
# --------------------------------------------------------------------------
def _atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def fingerprint(hp, n_train, n_classes):
    core = dict(hp, n_train=int(n_train), n_classes=int(n_classes), feat=(N_FRAMES, FEAT_DIM))
    return hashlib.sha1(json.dumps(core, sort_keys=True).encode()).hexdigest()[:12]


def train(arch, hp, data, run_dir, mean, std, X_val=None, y_val=None, fixed_epochs=None, log=print,
          log_file=None):
    """Train (or resume) in run_dir. data = (X_train float16, y_train int64).
    With X_val: early stopping on validation top-1. Without: runs fixed_epochs, keeps the last weights."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    X, y = data
    fp = fingerprint(dict(hp, **{k: arch[k] for k in ("proj", "hidden", "layers", "dropout")}), len(X), arch["n_classes"])
    epochs = fixed_epochs or hp["epochs"]
    if log_file:
        _p = log

        def log(msg, _q=_p):
            _q(msg)
            try:
                with open(log_file, "a", encoding="utf-8") as fh:
                    fh.write(time.strftime("%Y-%m-%d %H:%M:%S ") + str(msg) + "\n")
            except OSError:
                pass

    torch.manual_seed(hp["seed"])
    model = build_model(arch)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    gen = torch.Generator().manual_seed(hp["seed"])
    state = dict(epoch=0, step=0, best=-1.0, best_epoch=0, bad=0, history=[], fp=fp)
    last = run_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck["state"]["fp"] != fp:
            raise SystemExit(
                f"{run_dir} holds a run with different settings or data (fingerprint {ck['state']['fp']} vs {fp}). "
                "Use a new RUN_NAME, or restore the previous settings.")
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        gen.set_state(ck["gen"])
        state = ck["state"]
        log(f"resuming {run_dir.name} at epoch {state['epoch']}/{epochs} (best so far {state['best']:.4f})")
        if state["epoch"] >= epochs or (X_val is not None and state["bad"] >= hp["patience"]):
            log("this run is already finished.")
    log(f"model parameters: {n_params(model):,} | train clips: {len(X):,} | batch {hp['batch']} | epochs <= {epochs}")

    m_t, s_t = torch.from_numpy(mean), torch.from_numpy(std)
    y_t = torch.from_numpy(y)
    n = len(X)
    steps_per_epoch = math.ceil(n / hp["batch"])
    total_steps = steps_per_epoch * epochs
    warm = min(300, max(10, total_steps // 20))
    t_start = time.time()
    done_now = 0
    while state["epoch"] < epochs and not (X_val is not None and state["bad"] >= hp["patience"]):
        ep = state["epoch"] + 1
        t0 = time.time()
        model.train()
        perm = torch.randperm(n, generator=gen)
        tot_loss = tot_ok = 0.0
        for i in range(0, n, hp["batch"]):
            idx = perm[i:i + hp["batch"]].numpy()
            idx.sort()                                         # sorted reads from the float16 array
            xb = (torch.from_numpy(X[idx]).float() - m_t) / s_t
            xb = augment(xb, gen, hp["noise"], hp["mask_p"], hp["scale_lo"])
            yb = y_t[idx]
            for g_ in opt.param_groups:
                g_["lr"] = lr_at(state["step"], total_steps, hp["lr"], warm)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb, label_smoothing=hp["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            state["step"] += 1
            tot_loss += float(loss.detach()) * len(idx)
            tot_ok += float((logits.argmax(1) == yb).sum())
        state["epoch"] = ep
        rec = dict(epoch=ep, train_loss=tot_loss / n, train_acc=tot_ok / n, lr=opt.param_groups[0]["lr"])
        msg = f"epoch {ep:>3}/{epochs} | train loss {rec['train_loss']:.3f} acc {rec['train_acc']:.3f}"
        if X_val is not None:
            pv = predict_probs(model, X_val, mean, std)
            mv = metrics_from_probs(pv, y_val, arch["n_classes"])
            rec.update(val_top1=mv["top1"], val_top5=mv["top5"], val_loss=mv["loss"])
            msg += f" | val top1 {mv['top1']:.3f} top5 {mv['top5']:.3f} loss {mv['loss']:.3f}"
            if mv["top1"] > state["best"] + 1e-6:
                state.update(best=mv["top1"], best_epoch=ep, bad=0)
                _atomic_save(dict(model=model.state_dict(), epoch=ep, val=mv), run_dir / "best.pt")
                msg += "  *best*"
            else:
                state["bad"] += 1
        rec["seconds"] = time.time() - t0
        state["history"].append(rec)
        done_now += 1
        eta = (time.time() - t_start) / done_now * (epochs - ep) / 60
        log(msg + f" | {rec['seconds']:.0f}s | ETA <= {eta:.0f} min")
        with open(run_dir / "metrics.jsonl", "a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec) + "\n")
        _atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), gen=gen.get_state(), state=state), last)
    if X_val is None:
        _atomic_save(dict(model=model.state_dict(), epoch=state["epoch"]), run_dir / "final.pt")
    log(f"finished: {state['epoch']} epoch(s)" + (f", best validation top-1 {state['best']:.4f} at epoch {state['best_epoch']}"
                                                   if X_val is not None else ""))
    return model, state


# --------------------------------------------------------------------------
# export
# --------------------------------------------------------------------------
def export_bundle(out_dir, model, arch, mean, std, classes, extraction_cfg, metrics, threshold, notes, feature_spec=None):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out / "model.pt")
    (out / "model_config.json").write_text(json.dumps(dict(architecture="SignGRU", **arch), indent=2))
    (out / "preprocessing.json").write_text(json.dumps(dict(
        feature_spec=(feature_spec or {}).get("name", "sign_features_v3"), feature_spec_detail=feature_spec,
        n_frames=N_FRAMES, feat_dim=FEAT_DIM,
        standardize_mean=mean.tolist(), standardize_std=std.tolist(),
        extraction=extraction_cfg,
        note="Serving must sample frames and extract landmarks exactly like `extraction`, then apply "
             "sign_features_v3 and (x - mean) / std."), indent=1))
    (out / "labels.json").write_text(json.dumps(dict(classes=list(classes)), ensure_ascii=False, indent=1), encoding="utf-8")
    (out / "metrics.json").write_text(json.dumps(dict(metrics, confidence_threshold=threshold), indent=2))
    card = [
        "# SignSage FDMSE-ISL isolated-word model", "",
        f"* Architecture: SignGRU, {n_params(model):,} parameters, {arch['n_classes']} classes",
        f"* Input: {(feature_spec or {}).get('name', 'sign_features_v3')}, {N_FRAMES} x {FEAT_DIM}", f"* Confidence threshold: {threshold}", "",
        "## Data", "Trained on FDMSE-ISL (Patra et al., Pattern Analysis and Applications 28(3):148, 2025), "
        "used with the authors' permission. Weights follow the dataset's terms and do not inherit the code's MIT licence.", "",
        "## Results", "```json", json.dumps(metrics, indent=2), "```", "", "## Notes", notes, "",
        "## Limits", "Studio-recorded signers; webcam performance is unmeasured. Not validated for real-world use. "
        "Not promoted until the promotion gates in docs/MODELS_AND_DATA.md pass."]
    (out / "MODEL_CARD.md").write_text("\n".join(card), encoding="utf-8")
    return out


def load_bundle(bundle_dir):
    b = Path(bundle_dir)
    arch = json.loads((b / "model_config.json").read_text())
    pre = json.loads((b / "preprocessing.json").read_text())
    labels = json.loads((b / "labels.json").read_text(encoding="utf-8"))["classes"]
    met = json.loads((b / "metrics.json").read_text())
    model = build_model(arch)
    model.load_state_dict(torch.load(b / "model.pt", map_location="cpu"))
    model.eval()
    return dict(model=model, mean=np.array(pre["standardize_mean"], np.float32),
                std=np.array(pre["standardize_std"], np.float32), classes=labels,
                threshold=met.get("confidence_threshold"))


def predict_topk(bundle, feats, k=3):
    """feats: (60,122) sign_features_v3 array -> list of (word, probability), or [] if below the threshold."""
    p = predict_probs(bundle["model"], feats[None].astype(np.float32), bundle["mean"], bundle["std"])[0]
    top = np.argsort(-p)[:k]
    thr = bundle["threshold"]
    if thr is not None and p[top[0]] < thr:
        return []
    return [(bundle["classes"][i], float(p[i])) for i in top]


In [ ]:
# ============================ 4. LOAD DATA ============================
importlib.invalidate_caches()
import sp_train
importlib.reload(sp_train)

manifest = pd.read_csv(f"{META}/manifest.csv")
CLASSES = json.load(open(f"{META}/label_map.json", encoding="utf-8"))["classes"]
data = sp_train.load_dataset(FEATS, manifest)
S = data["stats"]
if S["converted"] < S["total_in_manifest"]:
    print(f"\nWARNING: only {S['converted']} of {S['total_in_manifest']} clips are converted to features yet. "
          "Training is possible but the numbers are NOT final.")

is_tr, is_va, is_te = (data["split"] == s for s in ("train", "validation", "test"))
X_tr, y_tr = data["X"][is_tr], data["y"][is_tr]
X_va, y_va = data["X"][is_va], data["y"][is_va]
X_te, y_te = data["X"][is_te], data["y"][is_te]
if MAX_TRAIN_CLIPS and len(X_tr) > MAX_TRAIN_CLIPS:
    keep = np.random.RandomState(SEED).permutation(len(X_tr))[:MAX_TRAIN_CLIPS]
    X_tr, y_tr = X_tr[keep], y_tr[keep]
    print(f"TRIAL RUN: using {len(X_tr)} training clips")
if len(X_tr) == 0 or len(X_va) == 0:
    raise SystemExit("Need training and validation clips. Convert more chunks in notebook 01 (cell 12).")
n_missing = S["n_classes"] - len(np.unique(y_tr))
if n_missing:
    print(f"NOTE: {n_missing} of {S['n_classes']} classes have no training clips yet (they cannot be learned).")

mean, std = sp_train.fit_standardizer(X_tr)
print()
QUALITY_WARNINGS = sp_train.quality_report(data, std)        # READ THIS before training
ARCH = dict(n_classes=S["n_classes"], proj=HIDDEN, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT)
HP = dict(epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING,
          patience=PATIENCE, noise=NOISE, mask_p=MASK_P, scale_lo=SCALE_LO, seed=SEED)
print(f"\nmemory for features: {data['X'].nbytes/2**20:.0f} MB (float16) | classes: {S['n_classes']}")

In [ ]:
# ============================ 5. TRAIN (resumable - rerun to continue) ============================
model, state = sp_train.train(ARCH, HP, (X_tr, y_tr), RUN_DIR, mean, std, X_val=X_va, y_val=y_va,
                              log_file=f"{RUN_DIR}/train.log")      # follow it: tail -f <RUN_DIR>/train.log
hist = pd.DataFrame(state["history"])
if len(hist):
    display_cols = [c for c in ("epoch", "train_loss", "train_acc", "val_top1", "val_top5", "val_loss", "seconds") if c in hist]
    print(hist[display_cols].tail(10).round(3).to_string(index=False))

In [ ]:
# ============================ 6. TEST ONCE + CONFIDENCE THRESHOLD + SAFETY CHECKS ============================
report_path = f"{RUN_DIR}/test_report.json"
if os.path.exists(report_path) and not FORCE_RETEST:
    REPORT = json.load(open(report_path))
    print("The test split was already evaluated for this run - showing the saved result (not re-testing).")
else:
    if len(X_te) == 0:
        raise SystemExit("No usable test clips yet.")
    best = torch.load(f"{RUN_DIR}/best.pt", map_location="cpu", weights_only=False)
    best_model = sp_train.build_model(ARCH); best_model.load_state_dict(best["model"]); best_model.eval()
    p_va = sp_train.predict_probs(best_model, X_va, mean, std)
    thr_prec, table = sp_train.calibrate_threshold(p_va, y_va, TARGET_PRECISION)
    rej = sp_train.rejection_checks(best_model, mean, std, X_va, thr_prec)      # non-sign probes
    probe_max = max(v["max_confidence"] for v in rej.values())
    # the final threshold must satisfy the precision target AND sit above what non-sign probes score
    thr = None if thr_prec is None else round(max(thr_prec, probe_max + 0.01), 2)
    p_te = sp_train.predict_probs(best_model, X_te, mean, std)
    m_te = sp_train.metrics_from_probs(p_te, y_te, ARCH["n_classes"])
    m_va = sp_train.metrics_from_probs(p_va, y_va, ARCH["n_classes"])
    rep = sp_train.per_class_report(p_te, y_te, CLASSES)
    rep.to_csv(f"{RUN_DIR}/test_per_class.csv", index=False)
    table.to_csv(f"{RUN_DIR}/calibration_validation.csv", index=False)
    acc_te = p_te.max(1) >= thr if thr is not None else np.zeros(len(y_te), bool)
    at_thr = dict(coverage=float(acc_te.mean()),
                  precision=float((p_te.argmax(1)[acc_te] == y_te[acc_te]).mean()) if acc_te.any() else None)
    REPORT = dict(run=RUN_NAME, best_epoch=best["epoch"], validation=m_va, test=m_te, confidence_threshold=thr,
                  precision_threshold=thr_prec, probe_max_confidence=probe_max,
                  test_at_threshold=at_thr, rejection_checks=rej, data=S, hyperparameters=HP, architecture=ARCH)
    json.dump(REPORT, open(report_path, "w"), indent=2)

T = REPORT["test"]
print(f"TEST (official signer-independent split, used once): top-1 {T['top1']:.3f} | top-5 {T['top5']:.3f} | "
      f"macro-recall {T['macro_recall']:.3f} | clips {T['n']} | classes present {T['n_classes_present']}")
print(f"best epoch {REPORT['best_epoch']} | validation top-1 {REPORT['validation']['top1']:.3f}")
print(f"confidence threshold for {TARGET_PRECISION:.0%} precision on validation: {REPORT['precision_threshold']}")
print(f"highest confidence reached by NON-sign probes: {REPORT['probe_max_confidence']:.3f}")
print(f"FINAL threshold = the larger of the two (+0.01): {REPORT['confidence_threshold']}")
print("on the test split at the final threshold:", REPORT["test_at_threshold"])
print("\nnon-sign probes (zeros, noise, frozen pose, 5 frames) - rejected by the precision threshold ALONE?")
for k, v in REPORT["rejection_checks"].items():
    print(f"  {k:<36} confidence {v['max_confidence']:.3f}  rejected: {v['rejected']}")
cov = (REPORT["test_at_threshold"] or {}).get("coverage")
if REPORT["confidence_threshold"] is None:
    print("  -> the precision target was not reachable: the model is too uncertain to deploy.")
elif cov is not None and cov < 0.5:
    print(f"  -> only {cov:.0%} of real test clips pass the final threshold: the model is not confident enough yet.")
print("These four probes are a minimum safety check, not proof that every non-sign input is rejected.")
pc = pd.read_csv(f"{RUN_DIR}/test_per_class.csv") if os.path.exists(f"{RUN_DIR}/test_per_class.csv") else None
if pc is not None:
    print("\nweakest 10 words on test:"); print(pc.head(10).round(2).to_string(index=False))

In [ ]:
# ============================ 7. FINAL RETRAIN ON ALL SIGNERS (for deployment) ============================
# Same recipe, fixed number of epochs (= the best epoch above), train + validation + test clips.
# The test number reported in cell 6 stays the one from the first run; this model cannot be tested again.
FINAL_DIR = f"{RUN_DIR}/final"
if RUN_FINAL_RETRAIN:
    X_all = np.concatenate([data["X"][is_tr], X_va, X_te]); y_all = np.concatenate([data["y"][is_tr], y_va, y_te])
    mean_all, std_all = sp_train.fit_standardizer(X_all)
    final_epochs = max(5, int(REPORT["best_epoch"]))
    final_model, fstate = sp_train.train(ARCH, HP, (X_all, y_all), FINAL_DIR, mean_all, std_all,
                                         fixed_epochs=final_epochs, log_file=f"{FINAL_DIR}/train.log")
else:
    print("skipped (RUN_FINAL_RETRAIN = False)")

In [ ]:
# ============================ 8. EXPORT ============================
ext_cfg = json.load(open(f"{META}/extraction_config.json"))
if RUN_FINAL_RETRAIN:
    exp_model, exp_mean, exp_std = final_model, mean_all, std_all
    note = (f"Deployment model: trained on all signers for {final_epochs} epochs. The reported metrics come from the "
            "first run (official split, test used once).")
else:
    exp_model = sp_train.build_model(ARCH)
    exp_model.load_state_dict(torch.load(f"{RUN_DIR}/best.pt", map_location="cpu", weights_only=False)["model"])
    exp_mean, exp_std = mean, std
    note = "Model trained on the official training split only; best epoch chosen on validation."
EXPORT_DIR = f"{RUN_DIR}/export"
sp_train.export_bundle(EXPORT_DIR, exp_model, ARCH, exp_mean, exp_std, CLASSES, ext_cfg,
                       dict(test=REPORT["test"], validation=REPORT["validation"], data=REPORT["data"],
                            test_at_threshold=REPORT["test_at_threshold"], rejection_checks=REPORT["rejection_checks"]),
                       REPORT["confidence_threshold"], note, feature_spec=FEATURE_SPEC)
b = sp_train.load_bundle(EXPORT_DIR)                      # reload check
probe = sp_train.predict_probs(b["model"], X_va[:4], b["mean"], b["std"])
print("export reloads OK ->", EXPORT_DIR, "| files:", sorted(os.listdir(EXPORT_DIR)))

## What next
- Read `export/MODEL_CARD.md` and `test_per_class.csv`; the weakest words show where more data or better features are needed.
- **Not promoted.** A model is promoted only after the gates in `docs/MODELS_AND_DATA.md` §6 (signer-independent result, calibrated threshold, rejection checks, extractor parity with the browser, licence review).
- Keep the exported folder: `model.pt`, `preprocessing.json` (includes the exact extraction settings serving must reproduce), `labels.json`, `metrics.json`.
- Cite the dataset: Patra et al., *Pattern Analysis and Applications* 28(3):148, 2025.